# Phenotype file QC — cleaning diagnostics

Reads the residualized `.pheno` files written by `05_phenotype_residualize.ipynb`
and checks whether the phenotype cleaning produced well-behaved values before
accumulation. An elevated full-sib cross-product that doesn't reflect genuine
shared environment is most likely driven by **outlier individuals who survived
the cleaning pipeline** (the `filter_plausible_range()` pre-filter and the
post-residual 5-SD trim) — either as isolated extremes, or as pairs of related
individuals who are both extreme in the same direction so the within-family
correlation slips under the population-level threshold.

## What this notebook checks

1. **Distribution shape** — each standardised residual file should be
   approximately N(0, 1) within each sex (it's literally standardised to
   mean 0 / SD 1 per sex by `residualize_phenotype()`). Bimodality,
   heavy tails, or skewness signal a cleaning failure.

2. **Tail heaviness / kurtosis** — excess kurtosis > 1–2 on the residualized
   phenotype means outliers survived the 5-SD trim (possible if outliers
   form a cluster rather than a clean tail).

3. **Outlier individuals** — any residual with |Y| > 5 should not exist after
   `residualize_phenotype()`'s trim step. Their presence means the trim ran
   on a different set (e.g. outlier_sd was effectively overridden, or the
   within-sex standardisation re-introduced extremes).

4. **N retained vs. expected** — a large drop from one covariate-set to the
   next (e.g. base → base_pcs) that isn't explained by missing covariates
   might indicate that one covariate column is pathological.

5. **Cross-covariate-set consistency** — the residualized distributions for
   `base` vs. `base_pcs_zip3_ses` should have the same approximate shape
   for any given phenotype. Large differences (beyond the expected R²
   reduction) are suspicious.

6. **Pairwise phenotype correlations on the residualized scale** — high
   pairwise correlations between phenotypes after residualization may indicate
   that the same individuals are outliers in multiple traits, amplifying the
   cross-product signal for all affected pairs.

Run `05_phenotype_residualize.ipynb` first; this notebook only reads its output.

## Setup

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Must match 05_phenotype_residualize.ipynb, which writes
# {RUN_DIR}/05_phenotypes/residualized -- NOT the retired
# 02_phenotype/{sample_set}/residualized layout this notebook used to assume.
PROJECT_DIR = "phenotypic_covariance_v9"
RUN_DIR     = "1kg_eur"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
R_DIR     = os.path.join(WORKSPACE_BUCKET, PROJECT_DIR, RUN_DIR)
PHENO_DIR = os.path.join(R_DIR, "05_phenotypes", "residualized")
PLOTS_DIR = os.path.join(R_DIR, "05_phenotypes", "qc_plots")
os.makedirs(PLOTS_DIR, exist_ok=True)

# outlier threshold: residualize_phenotype() trims at 5 SD -- anything larger
# in the output file means the trim failed or was bypassed
OUTLIER_SD = 5.0

# ── shared style (matches the other notebooks) ────────────────────────────────
SURFACE, INK, INK2, MUTED, GRID, AXIS = ("#fcfcfb", "#0b0b0b", "#52514e",
                                         "#898781", "#e1e0d9", "#c3c2b7")
RAMP = ["#86b6ef", "#3987e5", "#256abf", "#104281"]   # covariate sets are ordered
FONT_STACK = ["Helvetica Neue", "Helvetica", "Liberation Sans", "Nimbus Sans",
              "Arial", "DejaVu Sans"]
from matplotlib import font_manager as _fm
_have = {x.name for x in _fm.fontManager.ttflist}
FONT = next((f for f in FONT_STACK if f in _have), "DejaVu Sans")
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "figure.dpi": 120,
    "font.family": FONT, "font.size": 9,
    "mathtext.fontset": "custom", "mathtext.rm": FONT,
    "mathtext.it": f"{FONT}:italic", "mathtext.bf": f"{FONT}:bold",
    "axes.edgecolor": INK2, "axes.linewidth": 0.7, "axes.labelcolor": INK2,
    "axes.titlecolor": INK, "axes.spines.top": True, "axes.spines.right": True,
    "axes.grid": True, "axes.grid.which": "major", "axes.axisbelow": True,
    "grid.color": GRID, "grid.linewidth": 0.5, "grid.alpha": 0.7,
    "xtick.minor.visible": False, "ytick.minor.visible": False,
    "text.color": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "xtick.labelcolor": INK2, "ytick.labelcolor": INK2, "legend.frameon": False,
})
print(f"font in use: {FONT}")
print("PHENO_DIR:", PHENO_DIR)
print("exists:", os.path.isdir(PHENO_DIR))

## Discover phenotype files

In [ ]:
pheno_files = sorted(glob.glob(os.path.join(PHENO_DIR, "*.pheno")))
if not pheno_files:
    # the retired layout, in case an older run is what is actually on disk
    legacy = os.path.join(WORKSPACE_BUCKET, PROJECT_DIR, "02_phenotype")
    hint = ""
    if os.path.isdir(legacy):
        hint = ("\nAn older 02_phenotype/ tree exists: "
                + ", ".join(sorted(os.listdir(legacy))[:8])
                + "\nThat is the retired per-sample-set layout; point RUN_DIR at a "
                  "run under\nphenotypic_covariance_v9/ instead, or re-run "
                  "05_phenotype_residualize.ipynb.")
    raise AssertionError(
        f"no .pheno files under {PHENO_DIR}\n"
        f"exists: {os.path.isdir(PHENO_DIR)}{hint}")

file_meta = []
for path in pheno_files:
    stem = os.path.basename(path)[:-len(".pheno")]
    parts = stem.split("__", 2)
    if len(parts) != 3:
        continue
    phenotype_name, transform, covariate_set = parts
    file_meta.append(dict(phenotype_name=phenotype_name, transform=transform,
                           covariate_set=covariate_set, path=path))

phenotypes = sorted({m["phenotype_name"] for m in file_meta})
transforms = sorted({m["transform"] for m in file_meta})
covariate_sets = sorted({m["covariate_set"] for m in file_meta})

print(f"{len(file_meta)} files | {len(phenotypes)} phenotypes | "
      f"{len(transforms)} transforms | {len(covariate_sets)} covariate sets")
print("phenotypes:", phenotypes)

## Load all .pheno files

Each file has `FID IID Y` format with comment lines starting with `#`.
Load all into one long DataFrame for efficient aggregation.

In [ ]:
records = []
for m in file_meta:
    df = pd.read_csv(
        m["path"], sep=r"\s+", comment="#",
        dtype={"FID": str, "IID": str, "Y": float}
    )
    df["phenotype_name"] = m["phenotype_name"]
    df["transform"] = m["transform"]
    df["covariate_set"] = m["covariate_set"]
    records.append(df)

all_pheno = pd.concat(records, ignore_index=True)

print(f"Loaded {len(all_pheno):,} rows across {len(file_meta)} files")
print("Y range:", all_pheno["Y"].min(), "to", all_pheno["Y"].max())
all_pheno.head(3)

## 1. Outlier check — any |Y| > 5 SD

`residualize_phenotype()` trims at 5 SD before the within-sex standardisation.
No value in a correctly-cleaned .pheno file should exceed 5 SD — finding any
means the standardisation step re-expanded tails (possible if a sex stratum
has very few individuals after trimming, making the per-sex SD small) or the
trim was applied to a different population than the final file.

In [ ]:
outliers = all_pheno[all_pheno["Y"].abs() > OUTLIER_SD]
if outliers.empty:
    print(f"PASS: No |Y| > {OUTLIER_SD} in any phenotype file.")
else:
    print(f"FAIL: {len(outliers):,} rows with |Y| > {OUTLIER_SD}:")
    print(outliers.groupby(["phenotype_name", "transform", "covariate_set"])
          .agg(n_outliers=("Y", "count"), max_abs_Y=("Y", lambda x: x.abs().max()))
          .reset_index().sort_values("max_abs_Y", ascending=False))

## 2. N retained per combo

Compare across covariate sets and transforms. A large N drop that isn't
explained by missing covariate columns is suspicious.

In [ ]:
n_table = (
    all_pheno.groupby(["phenotype_name", "transform", "covariate_set"])
    .agg(n=("Y", "count"))
    .reset_index()
)

# pivot: phenotype x covariate_set (raw transform only)
cs_order = [cs for cs in ["base", "base_pcs", "base_pcs_zip3", "base_pcs_zip3_ses"]
            if cs in covariate_sets] or covariate_sets

n_pivot = (
    n_table[n_table["transform"] == ("invnorm" if "invnorm" in transforms else transforms[0])]
    .pivot(index="phenotype_name", columns="covariate_set", values="n")
    .reindex(columns=cs_order)
)
print("N retained per phenotype × covariate set (invnorm):")
display(n_pivot)

# flag any phenotype where N drops > 5% from base to full model
if "base" in n_pivot.columns and cs_order[-1] in n_pivot.columns:
    n_drop_pct = 100 * (n_pivot["base"] - n_pivot[cs_order[-1]]) / n_pivot["base"]
    flagged = n_drop_pct[n_drop_pct > 5].sort_values(ascending=False)
    if not flagged.empty:
        print(f"\nPhenotypes with >5% N drop from base to {cs_order[-1]}:")
        print(flagged)

## 3. Distribution shapes — density plots

Each distribution should look like N(0,1). Bimodality, heavy tails, or spikes
are red flags. One panel per phenotype; richest covariate set, invnorm transform.
A Q-Q plot against N(0,1) is shown alongside the density to make tail behaviour
visible.

In [ ]:
PLOT_TRANSFORM = "invnorm" if "invnorm" in transforms else transforms[0]
PLOT_CS = cs_order[-1]
MAX_PLOT_N = 30_000   # subsample cap for rendering only

sub = all_pheno[
    (all_pheno["transform"] == PLOT_TRANSFORM) &
    (all_pheno["covariate_set"] == PLOT_CS)
]

ncols = 4
nrows = (len(phenotypes) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows),
                          constrained_layout=True, squeeze=False)

x_ref = np.linspace(-4, 4, 200)
ref_density = stats.norm.pdf(x_ref)

for i, pheno in enumerate(phenotypes):
    ax = axes[i // ncols][i % ncols]
    vals = sub[sub["phenotype_name"] == pheno]["Y"].dropna().to_numpy()
    if len(vals) == 0:
        ax.set_visible(False)
        continue
    if len(vals) > MAX_PLOT_N:
        rng = np.random.default_rng(0)
        vals_plot = rng.choice(vals, MAX_PLOT_N, replace=False)
    else:
        vals_plot = vals

    ax.hist(vals_plot, bins=60, density=True, color="steelblue", alpha=0.6,
            label=f"N={len(vals):,}")
    ax.plot(x_ref, ref_density, color="firebrick", lw=1.2, ls="--", label="N(0,1)")
    ax.set_title(pheno, fontsize=7.5)
    ax.set_xlabel("Y", fontsize=7)
    ax.legend(fontsize=5.5, loc="upper left")
    ax.set_xlim(-6, 6)

for i in range(len(phenotypes), nrows * ncols):
    axes[i // ncols][i % ncols].set_visible(False)

fig.suptitle(f"Residualized phenotype distributions vs. N(0,1)\n"
             f"{RUN_DIR} | {PLOT_TRANSFORM} | {PLOT_CS}",
             fontsize=10)
fname = os.path.join(PLOTS_DIR, f"pheno_distributions_{PLOT_TRANSFORM}_{PLOT_CS}.png")
fig.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {fname}")

## 4. Q-Q plots against N(0,1)

Tighter diagnostic for tail behaviour. Deviation in the upper-right corner
means right-tail outliers survived cleaning; deviation in both corners means
heavy tails (kurtosis > 3). Either would inflate pair cross-products.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows),
                          constrained_layout=True, squeeze=False)

for i, pheno in enumerate(phenotypes):
    ax = axes[i // ncols][i % ncols]
    vals = sub[sub["phenotype_name"] == pheno]["Y"].dropna().to_numpy()
    if len(vals) == 0:
        ax.set_visible(False)
        continue

    # Q-Q: plot all points but subsample to speed up rendering
    if len(vals) > MAX_PLOT_N:
        rng = np.random.default_rng(1)
        vals_plot = np.sort(rng.choice(vals, MAX_PLOT_N, replace=False))
    else:
        vals_plot = np.sort(vals)

    theoretical = stats.norm.ppf(
        (np.arange(1, len(vals_plot) + 1)) / (len(vals_plot) + 1)
    )
    ax.scatter(theoretical, vals_plot, s=1, alpha=0.3, color="steelblue", rasterized=True)
    ref_line = np.array([-4, 4])
    ax.plot(ref_line, ref_line, color="firebrick", lw=1.0, ls="--")
    ax.set_title(pheno, fontsize=7.5)
    ax.set_xlabel("Theoretical N(0,1)", fontsize=6)
    ax.set_ylabel("Observed Y", fontsize=6)

for i in range(len(phenotypes), nrows * ncols):
    axes[i // ncols][i % ncols].set_visible(False)

fig.suptitle(f"Q-Q plots vs. N(0,1) — {RUN_DIR} | {PLOT_TRANSFORM} | {PLOT_CS}",
             fontsize=10)
fname = os.path.join(PLOTS_DIR, f"pheno_qq_{PLOT_TRANSFORM}_{PLOT_CS}.png")
fig.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {fname}")

## 5. Summary statistics — mean, SD, skewness, kurtosis per combo

After `residualize_phenotype()`'s per-sex standardisation:
- pooled mean ≈ 0 (exactly 0 within each sex; pooled mean depends on sex ratio)
- pooled SD ≈ 1 (exactly 1 within each sex; pooled SD depends on sex ratio)
- skewness: should be ~0 on invnorm, may be nonzero on raw
- excess kurtosis: should be ~0 for N(0,1); >1 signals outlier survivors

Phenotypes with excess kurtosis > 1 on the invnorm scale are flagged:
their fat tails can inflate the FS-range cross-products because high-Y pairs
contribute disproportionately to the bin mean.

In [ ]:
def moments(x):
    x = x.dropna()
    return pd.Series(dict(
        n=len(x),
        mean=float(x.mean()),
        sd=float(x.std()),
        skewness=float(stats.skew(x)),
        excess_kurtosis=float(stats.kurtosis(x)),   # Fisher definition: 0 for normal
        pct_gt3sd=float(100 * (x.abs() > 3).mean()),
        pct_gt4sd=float(100 * (x.abs() > 4).mean()),
        max_abs_Y=float(x.abs().max()),
    ))

moment_table = (
    all_pheno.groupby(["phenotype_name", "transform", "covariate_set"])["Y"]
    .apply(moments)
    .unstack()
    .reset_index()
)
moment_table = moment_table.sort_values(
    ["transform", "covariate_set", "excess_kurtosis"], ascending=[True, True, False]
)

print("Summary moments per combo (sorted by excess_kurtosis descending within each group):")
display(moment_table.round(4))

# Flag: invnorm excess kurtosis > 1
flagged_kurt = moment_table[
    (moment_table["transform"] == PLOT_TRANSFORM) &
    (moment_table["covariate_set"] == PLOT_CS) &
    (moment_table["excess_kurtosis"] > 1)
][["phenotype_name", "n", "excess_kurtosis", "pct_gt4sd", "max_abs_Y"]]
if not flagged_kurt.empty:
    print(f"\nFlagged (excess kurtosis > 1 on {PLOT_TRANSFORM}, {PLOT_CS}):")
    display(flagged_kurt.round(4))
else:
    print(f"\nNo phenotypes with excess kurtosis > 1 on {PLOT_TRANSFORM}, {PLOT_CS}.")

## 6. Raw vs. invnorm kurtosis comparison

Phenotypes where invnorm does NOT reduce excess kurtosis to near-0 are
problematic: the INT should always produce near-normal marginal distributions,
so residual kurtosis on the invnorm scale suggests outliers entered the INT
*after* the plausible-range filter and before the INT itself, or that the
INT was applied before (not after) some cleaning step. Check the order of
operations in `residualize_lib.R`.

In [ ]:
if "raw" in transforms and "invnorm" in transforms:
    kurt_raw = (
        moment_table[(moment_table["transform"] == "raw") &
                     (moment_table["covariate_set"] == PLOT_CS)]
        .set_index("phenotype_name")[["excess_kurtosis"]]
        .rename(columns={"excess_kurtosis": "kurt_raw"})
    )
    kurt_inv = (
        moment_table[(moment_table["transform"] == "invnorm") &
                     (moment_table["covariate_set"] == PLOT_CS)]
        .set_index("phenotype_name")[["excess_kurtosis"]]
        .rename(columns={"excess_kurtosis": "kurt_invnorm"})
    )
    kurt_both = kurt_raw.join(kurt_inv, how="inner")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(kurt_both["kurt_raw"], kurt_both["kurt_invnorm"], s=60, color="steelblue")
    for pheno, row in kurt_both.iterrows():
        ax.annotate(pheno, (row["kurt_raw"], row["kurt_invnorm"]),
                    fontsize=6, xytext=(3, 2), textcoords="offset points")
    lim_x = max(abs(kurt_both["kurt_raw"].max()), 0.1) * 1.15
    ax.axhline(0, color="grey", lw=0.5, ls="--")
    ax.axhline(1, color="tomato", lw=0.8, ls=":", label="kurtosis = 1 (flag threshold)")
    ax.set_xlabel("Excess kurtosis (raw scale)")
    ax.set_ylabel("Excess kurtosis (invnorm scale)")
    ax.set_title(f"Raw vs. invnorm excess kurtosis — {RUN_DIR}, {PLOT_CS}\n"
                 "(invnorm should push everything to ~0; points above red line need investigation)",
                 fontsize=8.5)
    ax.legend(fontsize=7)
    plt.tight_layout()
    fname = os.path.join(PLOTS_DIR, "kurtosis_raw_vs_invnorm.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {fname}")
else:
    print("Both 'raw' and 'invnorm' transforms required for this plot.")

## 7. Covariate-set consistency

The residualized distribution should get narrower (lower SD, closer to N(0,1))
as covariates are added, but the shape should stay similar. A distribution that
changes qualitatively (e.g. develops bimodality, or the kurtosis goes up) when
a new covariate block is added suggests a covariate column has unusual values
or collinear structure for this phenotype.

In [ ]:
cs_compare_transform = PLOT_TRANSFORM

ncols = len(cs_order)
nrows_p = (len(phenotypes) + 1) // 2
fig, axes = plt.subplots(len(phenotypes), ncols,
                          figsize=(3.5 * ncols, 2.5 * len(phenotypes)),
                          constrained_layout=True, squeeze=False)

for row_i, pheno in enumerate(phenotypes):
    for col_j, cs in enumerate(cs_order):
        ax = axes[row_i][col_j]
        vals = all_pheno[
            (all_pheno["phenotype_name"] == pheno) &
            (all_pheno["transform"] == cs_compare_transform) &
            (all_pheno["covariate_set"] == cs)
        ]["Y"].dropna().to_numpy()
        if len(vals) == 0:
            ax.set_visible(False)
            continue
        if len(vals) > MAX_PLOT_N:
            vals = np.random.default_rng(row_i * 100 + col_j).choice(vals, MAX_PLOT_N, replace=False)
        ax.hist(vals, bins=50, density=True, color="steelblue", alpha=0.65)
        ax.plot(x_ref, ref_density, color="firebrick", lw=1.0, ls="--")
        ax.set_title(f"{pheno}\n{cs.replace('base_', '')}", fontsize=6.5)
        ax.set_xlim(-6, 6)
        ax.set_xlabel("Y", fontsize=6)

fig.suptitle(f"Distribution by covariate set — {RUN_DIR}, {cs_compare_transform}",
             fontsize=10)
fname = os.path.join(PLOTS_DIR, f"dist_by_covset_{cs_compare_transform}.png")
fig.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {fname}")

## 8. Pairwise phenotype correlations on the residualized scale

After residualization and standardisation, phenotype correlations should
reflect genuine biology (e.g. height–weight, cholesterol–triglycerides).
Unexpectedly high correlations among phenotypes with no known biological
link may indicate the same individuals are outliers in multiple traits —
a sign that the cleaning step is not catching multi-trait extremes.

Uses the same set of IIDs in both files (inner join), subsampled for speed.

In [ ]:
MAX_CORR_N = 20_000

corr_sub = all_pheno[
    (all_pheno["transform"] == PLOT_TRANSFORM) &
    (all_pheno["covariate_set"] == PLOT_CS)
]

# pivot to wide: IID × phenotype
wide = corr_sub.pivot_table(index="IID", columns="phenotype_name", values="Y", aggfunc="first")
if len(wide) > MAX_CORR_N:
    wide = wide.sample(n=MAX_CORR_N, random_state=0)

cor_mat = wide.corr(method="pearson", min_periods=500)

# heatmap
n_pheno = len(phenotypes)
fig_size = max(7, 0.5 * n_pheno + 3)
fig, ax = plt.subplots(figsize=(fig_size, fig_size))
im = ax.imshow(cor_mat.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(len(cor_mat.columns)))
ax.set_yticks(range(len(cor_mat.index)))
ax.set_xticklabels(cor_mat.columns, rotation=60, ha="right", fontsize=7)
ax.set_yticklabels(cor_mat.index, fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.6, label="Pearson r")
ax.set_title(f"Residualized phenotype correlations — {RUN_DIR}, {PLOT_TRANSFORM}, {PLOT_CS}",
             fontsize=9)
plt.tight_layout()
fname = os.path.join(PLOTS_DIR, f"pheno_correlations_{PLOT_TRANSFORM}_{PLOT_CS}.png")
fig.savefig(fname, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {fname}")

# flag unexpectedly high off-diagonal correlations
CORR_FLAG = 0.5
cor_long = cor_mat.stack().reset_index()
cor_long.columns = ["pheno_x", "pheno_y", "r"]
cor_long = cor_long[cor_long["pheno_x"] < cor_long["pheno_y"]]  # upper triangle
high_corr = cor_long[cor_long["r"].abs() > CORR_FLAG].sort_values("r", ascending=False, key=abs)
if not high_corr.empty:
    print(f"\nPhenotype pairs with |r| > {CORR_FLAG}:")
    print(high_corr.to_string(index=False))
else:
    print(f"\nNo phenotype pairs with |r| > {CORR_FLAG}.")

## 9. Individuals with extreme residuals across multiple phenotypes

An individual who has |Y| > 3 in many phenotypes simultaneously is a
"super-outlier" — they likely had some data quality issue (e.g. measurement
error across many labs, or a pathological state not captured by the plausible
range filter). If this person has a sibling in the data who also shows a
correlated extreme (e.g. both have unusually high BMI AND high glucose), they
will inflate the FS-bin cross-products for all those phenotypes.

This cell counts how many phenotypes each individual is extreme in.

In [ ]:
EXTREME_SD = 3.0

extreme_counts = (
    all_pheno[
        (all_pheno["transform"] == PLOT_TRANSFORM) &
        (all_pheno["covariate_set"] == PLOT_CS) &
        (all_pheno["Y"].abs() > EXTREME_SD)
    ]
    .groupby("IID")["phenotype_name"]
    .agg(["count", lambda x: ", ".join(sorted(x))])
    .rename(columns={"count": "n_extreme_phenotypes", "<lambda_0>": "phenotypes"})
    .sort_values("n_extreme_phenotypes", ascending=False)
)

print(f"Individuals with |Y| > {EXTREME_SD} in at least one phenotype: "
      f"{len(extreme_counts):,}")
multi_extreme = extreme_counts[extreme_counts["n_extreme_phenotypes"] >= 3]
if not multi_extreme.empty:
    print(f"\nExtreme in >= 3 phenotypes ({len(multi_extreme)} individuals):")
    print(multi_extreme.head(20).to_string())
    print()
    print("These individuals may drive FS-bin inflation if their siblings are also extreme.")
    print("Consider adding an explicit multi-phenotype outlier filter before accumulation.")
else:
    print(f"\nNo individuals extreme in >= 3 phenotypes simultaneously.")

## Summary

Check the flagged items above:

- **Outliers with |Y| > 5** (section 1): shouldn't exist; indicates trim failed
- **High excess kurtosis on invnorm** (section 5): INT should force kurtosis to ~0;
  residual kurtosis means outliers entered AFTER the INT step
- **Super-outlier individuals** (section 9): extreme in many phenotypes simultaneously;
  if they have relatives in the data, they drive FS-bin inflation across traits
- **Bimodal distributions** (section 3 Q-Q/density plots): INT should fix this;
  if bimodality survives, the INT was applied to a subpopulation or the cleaning
  step produced two distinct clusters

Likely fix if cleaning is the issue: tighten the outlier trim (e.g. reduce
`outlier_sd` from 5 to 4 in `residualize_phenotype()`) or add a
multi-phenotype outlier filter to remove individuals who are extreme in
many phenotypes simultaneously. Both changes go in `residualize_lib.R`
and can be rerun without re-pulling from BigQuery (`run_residualization_from_tables()`).